# Đếm rep bằng YOLO-Pose — 3 bài, chạy được cả thư mục

Đếm số rep và tách các đoạn **đang tập / đang nghỉ** từ video, dùng `ultralytics`
YOLO11-pose thay cho MediaPipe. Đổi bài bằng biến `EXERCISE`:

| `EXERCISE` | Bài | Tín hiệu | Nguồn video |
|---|---|---|---|
| `'curl'` | Cuốn tạ đơn hai tay | `forearm_tilt` | Drive |
| `'overhead_extension'` | Đẩy tạ qua đầu | `wrist_above_nose` | Drive |
| `'push_up'` | Hít đất | `elbow_angle` | thư mục (`BATCH_DIR`) |

Phần lõi — trích keypoint, đếm rep, ghép hai tay, tách set, vẽ video — **dùng chung**.
Mỗi bài chỉ khác **tín hiệu 1 chiều** rút ra từ keypoint và **bộ ngưỡng** đi kèm.
Đặt `BATCH_DIR` thì notebook chạy hết mọi video trong thư mục và xuất bảng tổng hợp.

## Cách chạy trên Kaggle
1. Settings → **Accelerator: GPU T4 x2** (chạy CPU vẫn được nhưng chậm hơn ~10×).
2. Settings → **Internet: On** — bắt buộc, vì notebook tải video từ Google Drive
   bằng `gdown` và tải trọng số YOLO.
3. Chọn `EXERCISE` trong ô Cấu hình rồi Run All. Video tải tự động, không cần tạo Dataset.

## Kiến trúc 2 lượt (2-pass)

| Lượt | Việc | Chi phí |
|---|---|---|
| 1 | Chạy YOLO-pose một lần, lưu keypoint ra `.npz` | chậm (phần nặng nhất) |
| 2 | Tính tín hiệu → đếm rep → tách set → vẽ video | vài giây |

Tách như vậy để **chỉnh ngưỡng mà không phải chạy lại YOLO** — chạy lại ô phân tích là đủ.
Notebook push-up cũ gộp tất cả vào một vòng lặp, nên mỗi lần đổi ngưỡng phải suy luận
lại toàn bộ video.

## Ba tín hiệu, chọn theo bài

Cả ba đều được tính cho **từng tay**; profile của mỗi bài chỉ định dùng cái nào.

### `forearm_tilt` — độ nghiêng cẳng tay (bài cuốn tạ)

$$\mathrm{fore} \;=\; \frac{y_{\text{khuỷu}} - y_{\text{cổ tay}}}{\lVert \vec{p}_{\text{khuỷu}} - \vec{p}_{\text{cổ tay}} \rVert} \;\in\; [-1, +1]$$

Video `IMG_2425` quay rất gần, **đầu bị cắt khỏi khung** và vai hay bị che: keypoint vai
chỉ đạt độ tin cậy 0.09–0.93 và góc khuỷu chỉ hợp lệ ở 75–84% số mẫu, trong khi độ
nghiêng cẳng tay hợp lệ ở 89–92%. Nên bài đó **không** dùng góc khuỷu.

- Chỉ cần **khuỷu + cổ tay** — hai keypoint tin cậy nhất (~0.98).
- Tự chuẩn hoá theo chiều dài cẳng tay ⇒ không phụ thuộc độ phân giải, khoảng cách
  camera hay dáng người. (Notebook push-up cũ so ngưỡng pixel cứng `std < 100`, đổi độ
  phân giải là hỏng.)
- Duỗi thẳng tay xuống → `fore ≈ −1`; cuốn lên ngang ngực → `fore ≈ +0.7`.

### `wrist_above_nose` — cổ tay cao hơn mũi bao nhiêu (bài đẩy tạ qua đầu)

$$\mathrm{nosew} \;=\; \frac{y_{\text{mũi}} - y_{\text{cổ tay}}}{\lVert \vec{p}_{\text{vai T}} - \vec{p}_{\text{vai P}} \rVert}$$

Ở tư thế này hai tay nằm **sau và trên đầu**, nên khuỷu bị đầu che và tạ che: độ tin cậy
khuỷu chỉ 0.86–0.93, và góc khuỷu đo được nhiễu tới mức bộ đếm ra 11 rep ở đoạn thực tế
chỉ có 7. Ngược lại **mũi 0.965** và **vai 0.96–0.99** là hai mốc chắc nhất.

Lấy mũi làm gốc còn tự khử rung thân người và rung camera, vì cả hai làm mũi và cổ tay
dịch cùng chiều. Chia cho bề rộng vai để không phụ thuộc tỉ lệ. Kết quả tách rất mạnh:

| | khi đang tập | khi buông tay |
|---|---|---|
| `nosew` | **+0.38 … +1.01** | **−2.54 … −1.44** |

Đo trên video: đỉnh (duỗi thẳng tay) ≈ +1.0, đáy (hạ tạ sau gáy) ≈ +0.4, chu kỳ ~1.05 s.

### `elbow_angle` — góc khuỷu `vai–khuỷu–cổ tay` (bài hít đất)

Ở bộ video hít đất, camera đặt **thấp ngay phía trước**, nên cánh tay vung gần như song
song mặt phẳng ảnh và góc khuỷu dao động rất rõ **70° ↔ 170°**. Chi tiết cách chọn và
các bẫy đã gặp nằm ở mục *"Vì sao push-up dùng góc khuỷu"* cuối notebook.

### Vì sao không dùng chung một công thức

Ba tín hiệu trên cho thấy **tín hiệu phải chọn theo bài tập _và_ theo góc máy**, không
có công thức vạn năng. Riêng `elbow_angle` là ví dụ rõ nhất: nó **tốt nhất** cho hít đất
nhưng **hỏng** ở cả hai bài tạ, vì hai lý do khác nhau — video cuốn tạ mất keypoint vai
(đầu bị cắt khỏi khung), video đẩy qua đầu mất keypoint khuỷu (bị đầu che).

Bộ đếm rep bên dưới không quan tâm tín hiệu mang đơn vị gì; ngưỡng chỉ cần cùng đơn vị.

In [ ]:
!pip install -q ultralytics
!pip install -q --upgrade gdown

In [ ]:
import os, re, time, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from ultralytics import YOLO
import torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Torch', torch.__version__, '| device =', DEVICE)
print('OpenCV', cv2.__version__)

## Cấu hình

Các ngưỡng dưới đây được dò từ chính video `IMG_2425.MOV` (13.7 phút, 1080×1920, 30 fps):
trích keypoint toàn bộ video rồi soi phân bố tín hiệu. Nếu dùng video khác, chạy ô
**"Dò ngưỡng tự động"** ở gần cuối để lấy bộ ngưỡng mới.

In [ ]:
# ============================ CẤU HÌNH ============================
EXERCISE = 'push_up'   # <-- ĐỔI: 'curl' | 'overhead_extension' | 'push_up'

PROFILES = {
    'curl': dict(
        ten       = 'Cuốn tạ đơn hai tay (dumbbell curl)',
        url       = 'https://drive.google.com/file/d/1lHKOfD07OlZj9ZkPLHduob5-cfhqLM0B/view',
        video     = 'IMG_2425.MOV',
        signal    = 'forearm_tilt',       # đầu bị cắt khỏi khung -> không dùng được vai
        sig_label = 'độ nghiêng cẳng tay',
        sig_range = (-1.0, 1.0),          # dùng để vẽ đồ thị chạy trên video
        rep_hi    = +0.15,                # đi lên vượt mức này -> tính 1 rep
        rep_lo    = -0.55,                # tụt dưới mức này -> về trạng thái đáy
        min_amp   = 0.60,                 # biên độ đáy->đỉnh tối thiểu
    ),
    'overhead_extension': dict(
        ten       = 'Đẩy tạ qua đầu / duỗi tay sau gáy (overhead triceps extension)',
        url       = 'https://drive.google.com/file/d/12d40Fu77KQdDTFOs-n3_k-MPwEGRwJsw/view',
        video     = 'IMG_2427.MOV',
        signal    = 'wrist_above_nose',   # tay ở sau/trên đầu -> khuỷu bị che, góc khuỷu nhiễu
        sig_label = 'cổ tay trên mũi (lần bề rộng vai)',
        sig_range = (-3.0, 2.0),
        rep_hi    = 0.85,                 # duỗi thẳng tay lên
        rep_lo    = 0.60,                 # hạ tạ xuống sau gáy
        min_amp   = 0.30,
    ),
    'push_up': dict(
        ten       = 'Hít đất (push-up)',
        url       = None,                 # dùng BATCH_DIR / VIDEO_PATH, không tải Drive
        video     = None,
        signal    = 'elbow_angle',        # xem ghi chú bên dưới
        sig_label = 'góc khuỷu (độ)',
        sig_range = (0.0, 180.0),
        rep_hi    = 140.0,                # duỗi thẳng tay, thân lên cao
        rep_lo    = 100.0,                # hạ người xuống sát sàn
        min_amp   = 40.0,
        kp_conf   = 0.40,      # hạ ngưỡng: góc quay này keypoint yếu hơn
        combine   = 'mean',    # hai tay hít đất CỨNG NHẮC cùng pha -> xem ghi chú cuối
        min_period= 0.70,      # nhịp trung vị đo được 1.10 s; 0.70 chặn rep ma
    ),
}

assert EXERCISE in PROFILES, f'EXERCISE phải là một trong {list(PROFILES)}'
P = PROFILES[EXERCISE]
print('Bài tập:', P['ten'])

GDRIVE_URL = P['url']
VIDEO_NAME = P['video']

# Chạy hàng loạt: trỏ vào một thư mục thì notebook phân tích MỌI video trong đó và
# xuất một bảng tổng hợp, thay vì chỉ một video. Để None nếu chỉ chạy một video.
BATCH_DIR  = '/kaggle/input/push-up' if EXERCISE == 'push_up' else None

OUT_DIR    = '/kaggle/working'
# Tải vào /kaggle/temp: thư mục này KHÔNG bị tính vào dung lượng output của notebook
# (video ~925 MB, để trong /kaggle/working là chạm trần 20 GB khi commit).
DL_DIR     = '/kaggle/temp' if os.path.isdir('/kaggle') else OUT_DIR

VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv', '.m4v')

def list_videos(folder):
    if not folder or not os.path.isdir(folder):
        return []
    return sorted(os.path.join(folder, f) for f in os.listdir(folder)
                  if f.lower().endswith(VIDEO_EXT))

# Chạy hàng loạt vẫn phân tích chi tiết + vẽ video cho file ĐẦU TIÊN, để có đủ biểu đồ
# kiểm tra; bảng tổng hợp cả thư mục nằm ở mục "Chạy hàng loạt" cuối notebook.
BATCH_FILES = list_videos(BATCH_DIR)
if BATCH_DIR:
    assert BATCH_FILES, f'Không thấy video nào trong {BATCH_DIR}'
    VIDEO_PATH = BATCH_FILES[0]
    print(f'{len(BATCH_FILES)} video trong {BATCH_DIR}; xem chi tiết file đầu: '
          f'{os.path.basename(VIDEO_PATH)}')
else:
    VIDEO_PATH = f'{DL_DIR}/{VIDEO_NAME}'
# Nếu đã upload video thành Kaggle Dataset thì trỏ thẳng vào đó và bỏ qua ô tải:
# VIDEO_PATH = '/kaggle/input/ta-don/IMG_2425.MOV'

KPTS_NPZ   = f'{OUT_DIR}/kpts_{EXERCISE}.npz'      # cache keypoint của lượt 1 (file đầu)
OUT_VIDEO  = f'{OUT_DIR}/{EXERCISE}_output.mp4'

# --- Lượt 1: trích keypoint ---
MODEL_NAME = 'yolo11s-pose.pt'  # 's' thay vì 'n': độ tin cậy vai 0.09 -> 0.84 trên video này
IMGSZ      = 640
BOX_CONF   = 0.25    # ngưỡng tin cậy box người
KP_CONF    = P.get('kp_conf', 0.50)   # keypoint yếu hơn mức này coi như không có (NaN)
STRIDE     = 2       # suy luận mỗi N frame; 2 -> 15 Hz, thừa sức bắt rep dài ~2 s
BATCH      = 16      # gộp frame theo lô cho GPU

# --- Lượt 2: đếm rep ---
SMOOTH_WIN_S = 0.25   # cửa sổ làm mượt Savitzky-Golay (giây). Nhịp cuốn đo được ~1.4 s
MAX_GAP_S    = 0.50   # mất pose ngắn hơn mức này thì nội suy bù

# Ba ngưỡng này mang ĐƠN VỊ CỦA TÍN HIỆU nên lấy từ profile của bài đang chọn
REP_HI         = P['rep_hi']
REP_LO         = P['rep_lo']
REP_MIN_AMP    = P['min_amp']
REP_MIN_PERIOD = P.get('min_period', 0.50)  # khoảng cách tối thiểu giữa 2 rep (giây)
MERGE_WIN_S    = 0.40   # 2 tay lên cách nhau dưới mức này -> tính LÀ MỘT rep
COMBINE_ARMS   = P.get('combine', 'per_arm')  # 'per_arm' | 'mean'

# --- Lượt 2: tách set tập / nghỉ ---
SET_MAX_GAP_S     = 6.0   # 2 rep cách nhau lâu hơn mức này -> thuộc 2 set khác nhau
SET_PAD_S         = 1.5   # nới đầu/cuối set để bao trọn nhịp nâng đầu và cuối
SET_MIN_REPS      = 3     # set ít rep hơn mức này bị loại (động tác lẻ, gãi, chỉnh tạ)
SET_MAX_CADENCE_S = 3.0   # nhịp trung vị chậm hơn mức này thì không coi là set thật

# --- Lượt 2: vẽ video ---
RENDER_VIDEO = True
BLUR_FACE    = True   # làm mờ mặt nếu mặt lọt khung (2425 bị cắt đầu nên thường vô hiệu)
DRAW_PLOT    = True   # vẽ đồ thị tín hiệu chạy trên frame (bằng cv2, không dùng matplotlib)

# Ba tuỳ chọn dưới đây để file xuất ra đủ nhỏ cho Kaggle xem trước được.
# Xuất nguyên video 1080x1920 dài 13-14 phút ra ~425 MB; tab Output của Kaggle
# không dựng nổi preview cỡ đó và báo "Unexpected end of JSON input".
RENDER_ONLY_SETS = True   # chỉ vẽ các đoạn đang tập, bỏ hết đoạn nghỉ (~4x ngắn hơn)
RENDER_MARGIN_S  = 2.0    # chừa thêm vài giây trước/sau mỗi set
RENDER_SCALE     = 0.5    # thu nhỏ khung hình (0.5 -> 1/4 diện tích)
H264_CRF         = 28     # càng cao càng nhẹ và càng mờ; 23 = nét, 30 = nhẹ
# ==================================================================

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DL_DIR, exist_ok=True)
print('Video sẽ nằm ở:', VIDEO_PATH if not BATCH_DIR else f'(chạy hàng loạt) {BATCH_DIR}')

## Tải video từ Google Drive

Cần bật **Internet: On** trong Settings, và link Drive phải ở chế độ chia sẻ
*Anyone with the link*. Ô này bỏ qua nếu file đã có sẵn, nên chạy lại notebook
không tải lại từ đầu.

In [ ]:
import gdown

def drive_id(url):
    """Tách file id khỏi link Drive (chấp nhận cả dạng /file/d/<id>/view lẫn ?id=<id>)."""
    m = (re.search(r'/d/([A-Za-z0-9_-]{20,})', url)
         or re.search(r'[?&]id=([A-Za-z0-9_-]{20,})', url))
    assert m, f'Không tách được file id từ link: {url}'
    return m.group(1)


def fetch_video(url, dest, min_mb=10):
    if os.path.exists(dest) and os.path.getsize(dest) > min_mb * 1e6:
        print(f'Đã có sẵn: {dest} ({os.path.getsize(dest)/1e6:.0f} MB) — bỏ qua tải.')
    else:
        print('Đang tải từ Google Drive...')
        # Truyền id= thay vì url=: gdown >= 5 đã bỏ tham số fuzzy, còn id= thì
        # bản nào cũng nhận, nên notebook không phụ thuộc version gdown của Kaggle.
        gdown.download(id=drive_id(url), output=dest, quiet=False)

    assert os.path.exists(dest), (
        f'Tải thất bại. Kiểm tra: (1) Internet đã bật trong Settings? '
        f'(2) Link Drive đã chia sẻ "Anyone with the link"?')

    mb = os.path.getsize(dest) / 1e6
    # gdown gặp trang chặn của Drive sẽ lưu chính trang HTML đó thành file -> vài KB
    assert mb > min_mb, (
        f'File chỉ {mb:.2f} MB — gần như chắc chắn là trang lỗi HTML chứ không phải video. '
        f'Thường do quota tải của Drive, hoặc link chưa mở công khai.')

    cap = cv2.VideoCapture(dest)
    ok  = cap.isOpened() and cap.read()[0]
    cap.release()
    assert ok, f'OpenCV không đọc được {dest} — file hỏng hoặc codec không hỗ trợ.'
    print(f'OK: {dest} ({mb:.0f} MB)')

if BATCH_DIR:
    print('Chế độ chạy hàng loạt -> bỏ qua bước tải Drive.')
elif not GDRIVE_URL:
    print(f'Profile {EXERCISE} không có link Drive; tự trỏ VIDEO_PATH tới file của bạn.')
else:
    fetch_video(GDRIVE_URL, VIDEO_PATH)

## Hàm dùng chung

Chỉ số keypoint theo chuẩn COCO-17 mà YOLO-pose xuất ra.

In [ ]:
# COCO-17
NOSE, L_EYE, R_EYE, L_EAR, R_EAR = 0, 1, 2, 3, 4
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KNE, R_KNE, L_ANK, R_ANK = 11, 12, 13, 14, 15, 16
FACE_KPS = [NOSE, L_EYE, R_EYE, L_EAR, R_EAR]

SKELETON = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
            (11,13),(13,15),(12,14),(14,16)]

SIDES = {'L': (L_SHO, L_ELB, L_WRI), 'R': (R_SHO, R_ELB, R_WRI)}


def angle_at(a, b, c):
    """Góc tại đỉnh b (độ). a/b/c dạng (N,2). NaN nếu cạnh suy biến."""
    v1, v2 = a - b, c - b
    n1 = np.linalg.norm(v1, axis=-1)
    n2 = np.linalg.norm(v2, axis=-1)
    cos = np.sum(v1 * v2, -1) / np.clip(n1 * n2, 1e-6, None)
    out = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    out[(n1 < 1e-3) | (n2 < 1e-3)] = np.nan
    return out


def build_signals(kpts, kp_conf=KP_CONF):
    """(N,17,3) -> dict tín hiệu 1 chiều. Keypoint yếu thành NaN thay vì dùng bừa."""
    xy, kc = kpts[..., :2].astype(np.float64), kpts[..., 2]
    out = {}

    # Bề rộng vai: thước đo tỉ lệ ổn định nhất, không đổi theo tư thế tay.
    sho_w = np.linalg.norm(xy[:, L_SHO] - xy[:, R_SHO], axis=-1)
    sho_w[np.minimum(kc[:, L_SHO], kc[:, R_SHO]) < kp_conf] = np.nan
    sho_w = np.where(sho_w < 1e-3, np.nan, sho_w)
    # Khi pose sập, hai vai bị dồn về một điểm -> bề rộng vai gần 0 -> mọi tín hiệu
    # chia cho nó bắn lên hàng chục lần. Loại thẳng các mẫu lệch xa trung vị.
    if np.isfinite(sho_w).any():
        med = np.nanmedian(sho_w)
        sho_w[(sho_w < 0.45 * med) | (sho_w > 2.2 * med)] = np.nan

    for s, (i_sho, i_elb, i_wri) in SIDES.items():
        sho, elb, wri = xy[:, i_sho], xy[:, i_elb], xy[:, i_wri]

        # (a) Góc khuỷu — cần cả vai, mà vai là keypoint yếu nhất ở video này
        ang = angle_at(sho, elb, wri)
        ang[np.minimum.reduce([kc[:, i_sho], kc[:, i_elb], kc[:, i_wri]]) < kp_conf] = np.nan

        # (b) Độ nghiêng cẳng tay — chỉ cần khuỷu + cổ tay. ĐÂY LÀ TÍN HIỆU CHÍNH.
        #     Trục y của ảnh hướng xuống, nên (y_khuỷu - y_cổtay) > 0 <=> cổ tay CAO hơn khuỷu.
        fv   = elb - wri
        flen = np.linalg.norm(fv, axis=-1)
        fore = fv[:, 1] / np.clip(flen, 1e-6, None)
        fore[(np.minimum(kc[:, i_elb], kc[:, i_wri]) < kp_conf) | (flen < 1e-3)] = np.nan

        # (c) Cổ tay cao hơn MŨI bao nhiêu, đo bằng số lần bề rộng vai.
        #     Dùng cho bài đẩy tạ qua đầu: lấy đầu làm mốc nên tự khử rung thân
        #     và rung camera, và tránh hẳn keypoint khuỷu — thứ bị đầu che nên
        #     nhiễu nhất ở tư thế này (đo được: mũi 0.965 / vai 0.96–0.99 so với
        #     khuỷu 0.86–0.93).
        nose_w = (xy[:, NOSE, 1] - xy[:, i_wri, 1]) / sho_w
        nose_w[np.minimum(kc[:, NOSE], kc[:, i_wri]) < kp_conf] = np.nan

        out[f'ang{s}']   = ang
        out[f'fore{s}']  = fore
        out[f'nosew{s}'] = nose_w
    return out


def interp_short_gaps(x, max_gap):
    """Nội suy tuyến tính các khoảng NaN NGẮN. Khoảng dài giữ NaN (mất pose thật)."""
    x  = np.asarray(x, float).copy()
    ok = np.isfinite(x)
    if ok.sum() < 2:
        return x
    i = np.arange(len(x))
    filled = np.interp(i, i[ok], x[ok])
    run_id = np.cumsum(ok)                    # đánh số từng cụm NaN liên tiếp
    for g in np.unique(run_id[~ok]):
        m = (~ok) & (run_id == g)
        if m.sum() > max_gap:                 # cụm quá dài -> trả lại NaN
            filled[m] = np.nan
    filled[:np.argmax(ok)] = np.nan           # không ngoại suy hai đầu
    filled[len(x) - np.argmax(ok[::-1]):] = np.nan
    return filled


def smooth(x, win):
    """Savitzky-Golay bỏ qua NaN: lọc riêng từng đoạn liên tục để NaN không lan ra."""
    win = max(5, int(win) | 1)
    y   = np.full(len(x), np.nan, dtype=float)
    ok  = np.isfinite(x)
    if ok.sum() < 5:
        return np.asarray(x, float).copy()
    edges  = np.flatnonzero(np.diff(ok.astype(int)))
    bounds = np.concatenate([[0], edges + 1, [len(x)]])
    for a, b in zip(bounds[:-1], bounds[1:]):
        if not ok[a]:
            continue
        y[a:b] = savgol_filter(x[a:b], win, 2) if (b - a) >= win else x[a:b]
    return y

---
## Lượt 1 — Trích keypoint

Chạy YOLO-pose một lần trên toàn video, lưu ra `kpts_<bài>.npz`. Khi có nhiều người trong
khung thì lấy **box lớn nhất** (người gần camera nhất) chứ không lấy box có confidence
cao nhất — trên video này đã gặp frame có 2 box, và box lớn nhất mới là chủ thể.

Ô này bỏ qua nếu cache đã khớp video + `STRIDE` + model; không khớp thì tự trích lại.

In [ ]:
def extract_keypoints(video_path, model, stride=STRIDE, batch=BATCH,
                      imgsz=IMGSZ, conf=BOX_CONF, device=DEVICE):
    """Trả về (idx, kpts (N,17,3), bconf, meta). Frame không thấy người -> NaN."""
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f'Không mở được video: {video_path}'
    meta = dict(fps    = cap.get(cv2.CAP_PROP_FPS),
                width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print('{width}x{height}  {fps:.2f} fps  {total} frame  '
          '({dur:.1f}s)'.format(dur=meta['total'] / max(meta['fps'], 1e-9), **meta))

    idx, kpts, bconf = [], [], []
    buf, buf_i = [], []

    def flush():
        if not buf:
            return
        for bi, r in zip(buf_i, model.predict(buf, imgsz=imgsz, conf=conf,
                                              device=device, verbose=False)):
            n = 0 if r.boxes is None else len(r.boxes)
            if n:
                b    = r.boxes.xyxy.cpu().numpy()
                area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                j    = int(np.argmax(area))          # người to nhất = chủ thể
                xy   = r.keypoints.xy.cpu().numpy()[j]        # (17,2)
                kc   = r.keypoints.conf.cpu().numpy()[j]      # (17,)
                kpts.append(np.concatenate([xy, kc[:, None]], 1))
                bconf.append(float(r.boxes.conf.cpu().numpy()[j]))
            else:
                kpts.append(np.full((17, 3), np.nan, np.float32))
                bconf.append(0.0)
            idx.append(bi)
        buf.clear(); buf_i.clear()

    i, t0 = 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % stride == 0:
            buf.append(frame); buf_i.append(i)
            if len(buf) == batch:
                flush()
                if len(idx) % (batch * 20) == 0:
                    done = i / max(meta['total'], 1)
                    el   = time.time() - t0
                    print(f'  {i}/{meta["total"]} ({done*100:5.1f}%)  '
                          f'{el:5.0f}s  còn ~{el/max(done,1e-9)-el:.0f}s', flush=True)
        i += 1
    flush()
    cap.release()
    print(f'Xong {len(idx)} mẫu trong {time.time()-t0:.0f}s')
    return np.array(idx), np.array(kpts, np.float32), np.array(bconf, np.float32), meta


def cache_is_valid(path, video, stride, model_name):
    """Cache chỉ dùng lại được khi khớp CẢ video, stride và model.

    Khoá cache theo mỗi tên file là bẫy: đổi video mà quên xoá .npz thì notebook
    lặng lẽ phân tích keypoint của video cũ rồi vỡ ở bước vẽ vì lệch số frame.
    """
    if not os.path.exists(path):
        return False
    try:
        d = np.load(path)
        same = (str(d['src']) == str(video) and int(d['stride']) == int(stride)
                and str(d['model']) == str(model_name))
    except Exception as e:
        print('Cache hỏng, trích lại:', e)
        return False
    if not same:
        print(f'Cache của video/tham số khác -> trích lại '
              f'(cache: {str(d["src"])}, stride={int(d["stride"])}, model={str(d["model"])})')
    return same


if cache_is_valid(KPTS_NPZ, VIDEO_PATH, STRIDE, MODEL_NAME):
    print('Dùng lại cache:', KPTS_NPZ)
else:
    model = YOLO(MODEL_NAME)
    idx, kpts, bconf, meta = extract_keypoints(VIDEO_PATH, model)
    np.savez_compressed(KPTS_NPZ, idx=idx, kpts=kpts, bconf=bconf,
                        stride=STRIDE, src=VIDEO_PATH, model=MODEL_NAME, **meta)

_d    = np.load(KPTS_NPZ)
idx   = _d['idx']
kpts  = _d['kpts']
bconf = _d['bconf']
FPS   = float(_d['fps'])
W, H  = int(_d['width']), int(_d['height'])
TOTAL = int(_d['total'])
t     = idx / FPS                      # mốc thời gian của từng mẫu (giây)
SR    = FPS / float(_d['stride'])      # tần số lấy mẫu của tín hiệu (Hz)

print(f'{len(idx)} mẫu @ {SR:.1f} Hz | thấy người ở {np.isfinite(kpts[:,L_ELB,0]).mean()*100:.1f}% mẫu')

---
## Lượt 2a — Dựng tín hiệu

Nội suy các khoảng mất pose ngắn rồi làm mượt. Giữ **hai tay tách riêng** — bước đếm
ở dưới sẽ tự xử lý chuyện cuốn đồng thời hay xen kẽ.

In [ ]:
S = build_signals(kpts)

max_gap_n = max(1, int(round(MAX_GAP_S * SR)))
win_n     = max(5, int(round(SMOOTH_WIN_S * SR)) | 1)
print(f'làm mượt {win_n} mẫu ({win_n/SR:.2f}s), bù khoảng trống tối đa {max_gap_n} mẫu')

SIG_ALL = ['foreL', 'foreR', 'angL', 'angR', 'nosewL', 'nosewR']
for k in SIG_ALL:
    S[k + '_s'] = smooth(interp_short_gaps(S[k], max_gap_n), win_n)

# Chọn tín hiệu theo profile của bài tập.
SIG_KEY = {'forearm_tilt': 'fore', 'elbow_angle': 'ang',
           'wrist_above_nose': 'nosew'}[P['signal']]
SIG_L, SIG_R = S[SIG_KEY + 'L_s'], S[SIG_KEY + 'R_s']
with np.errstate(invalid='ignore'):
    SIG_M = np.nanmean(np.vstack([SIG_L, SIG_R]), axis=0)
SIGNALS = ({'Trung bình 2 tay': SIG_M} if COMBINE_ARMS == 'mean'
           else {'Tay trái': SIG_L, 'Tay phải': SIG_R})
print(f"Tín hiệu đếm: {P['signal']} ({P['sig_label']}), "
      f"ngưỡng HI={REP_HI} LO={REP_LO} biên độ tối thiểu={REP_MIN_AMP}")

for k in SIG_ALL:
    v = S[k]
    print(f'{k:7s} hợp lệ {np.isfinite(v).mean()*100:5.1f}%  '
          f'p05={np.nanpercentile(v,5):7.2f}  med={np.nanmedian(v):7.2f}  '
          f'p95={np.nanpercentile(v,95):7.2f}')

---
## Lượt 2b — Đếm rep

Dùng **trigger Schmitt** (ngưỡng kép có trễ) thay cho FSM 3 trạng thái của notebook
push-up cũ:

- Đang ở `đáy`, tín hiệu vượt `REP_HI` → **+1 rep**, chuyển sang `đỉnh`.
- Đang ở `đỉnh`, tín hiệu tụt dưới `REP_LO` → về `đáy`.

Hai chốt chặn nhiễu:

- **Biên độ**: chênh lệch đáy→đỉnh phải ≥ `REP_MIN_AMP`, loại rung tay khi đứng nghỉ.
- **Chu kỳ**: hai rep phải cách nhau ≥ `REP_MIN_PERIOD` giây.

Khác biệt so với notebook cũ: ở đó điều kiện đếm dùng `velocity` lấy từ 2 mẫu cuối
nhưng `avg_angle` lại là trung bình 15 mẫu — hai đại lượng lệch pha nhau, nên phải
vừa đúng ngưỡng góc vừa đúng dấu vận tốc mới đếm, dễ bỏ sót rep. Ở đây chỉ dựa vào
mức tín hiệu đã làm mượt, không cần vận tốc.

### Ghép hai tay

Mỗi tay được đếm **độc lập**, sau đó hai rep của hai tay cách nhau dưới `MERGE_WIN_S`
giây được gộp làm một. Cách này tự đúng cho cả ba kiểu tập, không cần khai báo trước:

| Kiểu tập | Kết quả |
|---|---|
| Hai tay đồng thời (cuốn tạ 2 tay, đẩy tạ qua đầu) | 2 rep trùng thời điểm → gộp thành 1 |
| Xen kẽ từng tay | lệch nhau nửa chu kỳ (~0.7 s) > `MERGE_WIN_S` → đếm riêng |
| Chỉ một tay | tay kia im, không sinh rep → đếm đúng tay đang tập |

Lấy **trung bình hai tay rồi mới đếm** thì hỏng ở kiểu thứ ba: trong `IMG_2425`, đoạn
780–785 s chỉ tay trái cuốn, tín hiệu trung bình chỉ lên tới ≈ +0.05 nên không vượt
`REP_HI` và mất trắng các rep đó.

In [ ]:
def detect_reps(t, s, arm, hi=REP_HI, lo=REP_LO,
                min_amp=REP_MIN_AMP, min_period=REP_MIN_PERIOD):
    """Trigger Schmitt trên tín hiệu đã làm mượt -> danh sách rep của MỘT tay."""
    reps, state = [], None
    trough, trough_t = np.inf, np.nan
    last_rep_t = -np.inf

    for i in range(len(s)):
        v = s[i]
        if not np.isfinite(v):
            state = None                      # mất pose -> chờ khoá lại trạng thái
            continue
        if state is None:
            state = 'up' if v > hi else 'down'
            trough, trough_t = v, t[i]
            continue

        if state == 'down':
            if v < trough:
                trough, trough_t = v, t[i]
            if v > hi:
                amp = v - trough
                if amp >= min_amp and (t[i] - last_rep_t) >= min_period:
                    reps.append(dict(t_top=t[i], t_bottom=trough_t,
                                     bottom=trough, top=v, amp=amp, arm=arm))
                    last_rep_t = t[i]
                state = 'up'
        else:                                 # state == 'up'
            if v < lo:
                state, trough, trough_t = 'down', v, t[i]
    return reps


def merge_arms(reps_l, reps_r, win=MERGE_WIN_S):
    """Gộp rep của hai tay xảy ra gần nhau thành một (cuốn đồng thời)."""
    out = []
    for r in sorted(reps_l + reps_r, key=lambda r: r['t_top']):
        prev = out[-1] if out else None
        if prev and prev['arm'] != r['arm'] and r['t_top'] - prev['t_top'] <= win:
            prev['arm']      = 'LR'
            prev['t_top']    = (prev['t_top'] + r['t_top']) / 2
            prev['t_bottom'] = min(prev['t_bottom'], r['t_bottom'])
            prev['amp']      = max(prev['amp'], r['amp'])
            prev['top']      = max(prev['top'], r['top'])
        else:
            out.append(dict(r))
    return out


if COMBINE_ARMS == 'mean':
    # Bài mà hai tay bị ràng buộc cứng cùng pha (hít đất): đếm trên TRUNG BÌNH
    # hai tay. Đếm tách rồi gộp sẽ sinh rep ma khi pose gán nhầm trái/phải —
    # đo được 120 rep ở nơi thực tế có ~97.
    all_reps = detect_reps(t, SIG_M, 'M')
    TOTAL_REPS = len(all_reps)
    print(f'trung bình hai tay -> {TOTAL_REPS} rep')
else:
    reps_L = detect_reps(t, SIG_L, 'L')
    reps_R = detect_reps(t, SIG_R, 'R')
    all_reps = merge_arms(reps_L, reps_R)
    TOTAL_REPS = len(all_reps)
    n_both = sum(1 for r in all_reps if r['arm'] == 'LR')
    print(f'tay trái {len(reps_L)} + tay phải {len(reps_R)}  ->  {TOTAL_REPS} rep sau khi gộp')
    print(f'  {n_both} rep hai tay đồng thời, {TOTAL_REPS - n_both} rep một tay')

---
## Lượt 2c — Tách đoạn TẬP / NGHỈ

Notebook push-up cũ nhận diện nghỉ bằng độ lệch chuẩn vị trí vai **hoặc** vai nằm cao
trong khung (`low_movement or high_position`), với ngưỡng pixel cứng. Điều đó khiến
người tập chậm và đều bị coi là đang nghỉ và **mất rep**.

Ở đây đoạn tập được suy ra **trực tiếp từ các rep đã đếm được**: rep là bằng chứng
đang tập, nên không thể mâu thuẫn với bộ đếm. Các rep liên tiếp cách nhau dưới
`SET_MAX_GAP_S` giây được gom thành một **set**; ngoài set là **nghỉ**.

Hai bộ lọc để không biến động tác lẻ thành "set":

- **Số rep**: dưới `SET_MIN_REPS` thì bỏ (gãi lưng, nhấc tạ lên chỉnh tay…).
- **Nhịp**: set thật có tiết tấu đều. Nhịp trung vị chậm hơn `SET_MAX_CADENCE_S`
  thì loại. Trên `IMG_2425`, lọc nhịp cắt 17 cụm ứng viên xuống còn 9 set thật.

Hai bộ lọc này quan trọng hơn nhiều ở `IMG_2427`: video đó phần lớn là đứng tạo dáng,
chống hông, đưa tay lên ngực — toàn những động tác đơn lẻ dễ vượt ngưỡng biên độ nhưng
không có tiết tấu.

In [ ]:
def group_sets(reps, max_gap=SET_MAX_GAP_S, pad=SET_PAD_S,
               min_reps=SET_MIN_REPS, max_cadence=SET_MAX_CADENCE_S):
    """Gom rep liên tiếp thành set. Trả list dict(start, end, n_reps, reps, dur)."""
    if not reps:
        return []
    groups, cur = [], [reps[0]]
    for r in reps[1:]:
        if r['t_top'] - cur[-1]['t_top'] <= max_gap:
            cur.append(r)
        else:
            groups.append(cur)
            cur = [r]            # list MỚI — đừng dùng cur.clear(), mọi group sẽ trỏ chung
    groups.append(cur)

    sets = []
    for g in groups:
        if len(g) < min_reps:
            continue
        gaps = np.diff([r['t_top'] for r in g])
        if len(gaps) and np.median(gaps) > max_cadence:   # không có tiết tấu -> không phải set
            continue
        sets.append(dict(start=max(0.0, min(r['t_bottom'] for r in g) - pad),
                         end=max(r['t_top'] for r in g) + pad,
                         n_reps=len(g), reps=list(g)))

    # nới biên bằng pad có thể làm 2 set liền nhau chồng lên nhau -> nhập lại làm một
    out = []
    for s in sets:
        if out and s['start'] <= out[-1]['end']:
            out[-1]['end']    = max(out[-1]['end'], s['end'])
            out[-1]['n_reps'] += s['n_reps']
            out[-1]['reps']   += s['reps']
        else:
            out.append(s)
    for s in out:
        s['dur'] = s['end'] - s['start']
    return out


SETS = group_sets(all_reps)

# Nhãn TẬP/NGHỈ cho từng mẫu
is_working = np.zeros(len(t), bool)
set_id     = np.full(len(t), -1, int)
for k, st in enumerate(SETS):
    m = (t >= st['start']) & (t <= st['end'])
    is_working[m] = True
    set_id[m]     = k

# Số rep tích luỹ tại mỗi mẫu (để hiển thị trên video)
rep_times   = np.array([r['t_top'] for r in all_reps]) if all_reps else np.zeros(0)
cum_reps    = np.searchsorted(rep_times, t, side='right')
# Số rep tích luỹ TRONG set hiện tại
reps_in_set = np.zeros(len(t), int)
for k, st in enumerate(SETS):
    m = set_id == k
    tops = np.array([r['t_top'] for r in st['reps']])
    reps_in_set[m] = np.searchsorted(tops, t[m], side='right')

work_s = is_working.sum() / SR
rest_s = (~is_working).sum() / SR
print(f'{len(SETS)} set | tập {work_s:.0f}s ({work_s/(work_s+rest_s)*100:.1f}%) | '
      f'nghỉ {rest_s:.0f}s')

In [ ]:
rows = []
for k, st in enumerate(SETS, 1):
    gaps = np.diff([r['t_top'] for r in st['reps']])
    rows.append({
        'Set': k,
        'Bắt đầu': time.strftime('%M:%S', time.gmtime(st['start'])),
        'Kết thúc': time.strftime('%M:%S', time.gmtime(st['end'])),
        'Thời lượng (s)': round(st['dur'], 1),
        'Số rep': st['n_reps'],
        'Nhịp TB (s/rep)': round(float(gaps.mean()), 2) if len(gaps) else np.nan,
        'Biên độ TB': round(float(np.mean([r['amp'] for r in st['reps']])), 2),
    })
df_sets = pd.DataFrame(rows)

# Thời gian nghỉ giữa các set
if len(SETS) > 1:
    df_sets['Nghỉ trước set (s)'] = [np.nan] + [
        round(SETS[i]['start'] - SETS[i-1]['end'], 1) for i in range(1, len(SETS))]

df_sets.to_csv(f'{OUT_DIR}/sets.csv', index=False)
pd.DataFrame(all_reps).drop(columns=['reps'], errors='ignore') \
    .to_csv(f'{OUT_DIR}/reps.csv', index=False)
display(df_sets)

# Con số chính là rep NẰM TRONG SET, không phải mọi nhịp phát hiện được: một cú
# nhấc tạ lẻ khi chỉnh tư thế vẫn vượt ngưỡng biên độ nhưng không phải rep của set nào.
REPS_IN_SETS = sum(st['n_reps'] for st in SETS)
print(f'\nTỔNG: {REPS_IN_SETS} rep trong {len(SETS)} set')
if TOTAL_REPS > REPS_IN_SETS:
    print(f'({TOTAL_REPS - REPS_IN_SETS} nhịp lẻ bị loại vì không đủ {SET_MIN_REPS} rep '
          f'liên tiếp hoặc không thành nhịp — thường là chỉnh tạ, tạo dáng)')

---
## Biểu đồ kiểm tra

Đây là chỗ để **mắt thường xác nhận** ngưỡng đặt đúng: mỗi chấm đỏ phải rơi đúng đỉnh
một nhịp cuốn, và mỗi dải xanh phải trùng một loạt nhịp liên tiếp.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(20, 10), sharex=True,
                       gridspec_kw=dict(height_ratios=[3, 2, 1]))

# (1) Tín hiệu đếm + rep + set
a = ax[0]
for st in SETS:
    a.axvspan(st['start'], st['end'], color='tab:green', alpha=.13, lw=0)
for name, s in SIGNALS.items():
    a.plot(t, s, lw=.9, label=name)
if all_reps:
    a.plot([r['t_top'] for r in all_reps], [r['top'] for r in all_reps],
           'v', color='crimson', ms=5, label=f'rep ({TOTAL_REPS})')
a.axhline(REP_HI, color='crimson', ls='--', lw=.8)
a.axhline(REP_LO, color='steelblue', ls='--', lw=.8)
a.text(t[-1], REP_HI, ' REP_HI', va='bottom', ha='right', color='crimson', fontsize=8)
a.text(t[-1], REP_LO, ' REP_LO', va='top', ha='right', color='steelblue', fontsize=8)
a.set_ylabel(P['sig_label'])
a.set_title(f"{P['ten']} — {TOTAL_REPS} rep / {len(SETS)} set — dải xanh = đang tập")
a.legend(loc='upper right', fontsize=8); a.grid(alpha=.25)

# (2) Góc khuỷu (đối chiếu)
ax[1].plot(t, S['angL_s'], lw=.7, label='khuỷu trái')
ax[1].plot(t, S['angR_s'], lw=.7, label='khuỷu phải', alpha=.75)
ax[1].set_ylabel('góc khuỷu (°)')
ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(alpha=.25)

# (3) Chất lượng phát hiện
ax[2].fill_between(t, 0, is_working.astype(float), step='mid',
                   color='tab:green', alpha=.5, label='đang tập')
ax[2].plot(t, bconf, lw=.5, color='gray', label='box conf')
ax[2].set_ylabel('trạng thái'); ax[2].set_xlabel('thời gian (giây)')
ax[2].legend(loc='upper right', fontsize=8); ax[2].grid(alpha=.25)

plt.tight_layout(); plt.savefig(f'{OUT_DIR}/analysis.png', dpi=100); plt.show()

In [ ]:
# Phóng to set dài nhất để soi từng nhịp
if SETS:
    st = max(SETS, key=lambda s: s['n_reps'])
    m  = (t >= st['start'] - 2) & (t <= st['end'] + 2)
    plt.figure(figsize=(20, 4))
    for name, s in SIGNALS.items():
        plt.plot(t[m], s[m], lw=1.4, marker='.', ms=3, label=name)
    for r in st['reps']:
        plt.axvline(r['t_top'], color='crimson', lw=.8, alpha=.6)
    plt.axhline(REP_HI, color='crimson', ls='--', lw=.8)
    plt.axhline(REP_LO, color='steelblue', ls='--', lw=.8)
    plt.title(f"Set dài nhất: {st['n_reps']} rep, {st['dur']:.0f}s "
              f"(vạch đỏ = thời điểm đếm)")
    plt.xlabel('giây'); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

---
## Dò ngưỡng tự động

Chạy ô này khi **đổi sang video khác**: nó lấy mức đáy/đỉnh thực tế rồi đề xuất
`REP_HI` / `REP_LO` đặt vào 1/3 và 2/3 khoảng dao động. Chép kết quả ngược lên ô
Cấu hình rồi chạy lại từ mục *Lượt 2a*.

**Chỉ lấy phân vị trong các đoạn đang tập.** Tính trên toàn video là sai ở những bài mà
nhịp rep chỉ là **gợn sóng nhỏ nằm trên một tư thế nền lớn** — đúng trường hợp bài đẩy
tạ qua đầu: `nosew` chạy từ −2.5 (buông tay) lên +1.0 (giơ qua đầu), trong khi nhịp rep
chỉ dao động +0.4 ↔ +1.0. Lấy phân vị toàn video sẽ đề xuất ngưỡng nằm giữa −1.3 và
−0.15, tức là chỉ đếm mỗi lần **giơ tay lên**, mất sạch các rep bên trong.

In [ ]:
# Chỉ lấy phần LÕI của mỗi set: từ rep thứ hai đến rep cuối. Bỏ `SET_PAD_S` và nhịp
# đầu tiên, vì hai chỗ đó còn chứa lúc mới nâng tạ lên — chính đoạn kéo lệch phân vị.
core = np.zeros(len(t), bool)
for st in SETS:
    if len(st['reps']) >= 2:
        core |= (t >= st['reps'][1]['t_top']) & (t <= st['reps'][-1]['t_top'])

in_sets = bool(core.any())
mask    = core if in_sets else np.ones(len(t), bool)
scope   = 'phần lõi của các set' if in_sets else 'TOÀN VIDEO'
if not in_sets:
    print('Chưa nhận được set nào -> lấy phân vị toàn video, độ tin cậy thấp.\n')

sig_act = np.concatenate([s[mask & np.isfinite(s)] for s in SIGNALS.values()])
lo_lvl, hi_lvl = np.percentile(sig_act, [5, 95])
span = hi_lvl - lo_lvl
print(f'Phạm vi tính: {scope}  ({mask.sum()} / {len(t)} mẫu)')
print(f'Đáy  (p5)  = {lo_lvl:+.2f}')
print(f'Đỉnh (p95) = {hi_lvl:+.2f}')
print(f'Biên độ    = {span:.2f}')
print('\nĐề xuất:')
print(f'  rep_lo  = {lo_lvl + span/3:+.2f}')
print(f'  rep_hi  = {lo_lvl + 2*span/3:+.2f}')
print(f'  min_amp = {span*0.5:.2f}')

full = np.concatenate([s[np.isfinite(s)] for s in SIGNALS.values()])
span_full = np.subtract(*np.percentile(full, [95, 5]))
if in_sets and span_full > 2.5 * span:
    print(f'\nLưu ý: biên độ toàn video ({span_full:.2f}) lớn hơn nhiều biên độ trong set '
          f'({span:.2f}) — nhịp rep là gợn sóng trên một tư thế nền. Đừng đặt ngưỡng '
          f'theo phân bố toàn video.')
if span < 0.15 * abs(P['sig_range'][1] - P['sig_range'][0]):
    print('\nCẢNH BÁO: biên độ quá nhỏ so với thang tín hiệu — pose có thể đang nhiễu, '
          'hoặc tín hiệu không hợp với bài này. Xem biểu đồ trước khi tin số rep.')

---
## Xuất video có chú thích

Đọc lại video và vẽ chồng kết quả. Vì keypoint đã có sẵn từ lượt 1 nên ô này chỉ tốn
thời gian giải mã + mã hoá, không chạy lại mạng.

Đồ thị chạy được vẽ bằng `cv2.polylines`. Notebook push-up cũ dựng một `matplotlib.Figure`
**cho mỗi frame** — cực chậm, và `fig.canvas.tostring_rgb()` mà nó dùng đã bị gỡ khỏi
Matplotlib từ bản 3.10 nên ô đó giờ sẽ lỗi.

In [ ]:
def blur_face(img, kp, w, h, kp_conf=KP_CONF):
    """Làm mờ vùng mặt nếu đủ keypoint mặt tin cậy. Không có mặt trong khung -> bỏ qua."""
    pts = np.array([kp[i, :2] for i in FACE_KPS if kp[i, 2] >= kp_conf])
    if len(pts) < 2:
        return
    pad_x, pad_y = 40, 60
    x0 = max(0, int(pts[:, 0].min()) - pad_x); x1 = min(w, int(pts[:, 0].max()) + pad_x)
    y0 = max(0, int(pts[:, 1].min()) - pad_y); y1 = min(h, int(pts[:, 1].max()) + pad_y)
    roi = img[y0:y1, x0:x1]
    if roi.size == 0:
        return
    k = max(3, int(min(roi.shape[:2]) * 0.4) | 1)
    img[y0:y1, x0:x1] = cv2.GaussianBlur(roi, (k, k), 0)


def draw_skeleton(img, kp, kp_conf=KP_CONF):
    for i, j in SKELETON:
        if kp[i, 2] >= kp_conf and kp[j, 2] >= kp_conf:
            cv2.line(img, tuple(kp[i, :2].astype(int)), tuple(kp[j, :2].astype(int)),
                     (245, 245, 245), 2, cv2.LINE_AA)
    for i in range(17):
        if kp[i, 2] >= kp_conf:
            c = (0, 200, 255) if i in (L_ELB, R_ELB, L_WRI, R_WRI) else (200, 200, 200)
            cv2.circle(img, tuple(kp[i, :2].astype(int)), 5, c, -1, cv2.LINE_AA)


def draw_trace(img, k, box, n_pts=150):
    """Đồ thị tín hiệu 2 tay đang chạy, vẽ bằng cv2 (nhanh hơn matplotlib rất nhiều)."""
    x0, y0, bw, bh = box
    cv2.rectangle(img, (x0, y0), (x0 + bw, y0 + bh), (25, 25, 25), -1)
    lo, hi = P['sig_range']
    to_y = lambda v: y0 + bh * (1 - (np.clip(v, lo, hi) - lo) / (hi - lo))
    for lvl, col in ((REP_HI, (60, 60, 235)), (REP_LO, (235, 160, 60))):
        y = int(to_y(lvl))
        cv2.line(img, (x0, y), (x0 + bw, y), col, 1, cv2.LINE_AA)

    a = max(0, k - n_pts)
    for sig, col in ((SIG_L, (255, 170, 60)), (SIG_R, (90, 90, 255))):
        seg = sig[a:k + 1]
        if len(seg) < 2:
            continue
        xs = np.linspace(x0, x0 + bw, len(seg))
        ys = to_y(seg)
        ok = np.isfinite(ys)
        if ok.sum() >= 2:
            pts = np.stack([xs[ok], ys[ok]], 1).astype(np.int32)
            cv2.polylines(img, [pts], False, col, 2, cv2.LINE_AA)


def draw_hud(img, k, w):
    """Bảng thông tin: tổng rep, set, trạng thái tập/nghỉ."""
    working = is_working[k]
    sid     = set_id[k]
    pw, ph  = 470, 210
    x0, y0  = w - pw - 30, 30
    panel   = img[y0:y0+ph, x0:x0+pw].copy()
    cv2.rectangle(panel, (0, 0), (pw, ph), (0, 0, 0), -1)
    img[y0:y0+ph, x0:x0+pw] = cv2.addWeighted(panel, .55, img[y0:y0+ph, x0:x0+pw], .45, 0)

    col   = (80, 230, 80) if working else (110, 110, 240)
    label = f'DANG TAP - Set {sid+1}' if working else 'DANG NGHI'
    F     = cv2.FONT_HERSHEY_SIMPLEX
    cv2.rectangle(img, (x0, y0), (x0 + pw, y0 + ph), col, 2)
    cv2.putText(img, label, (x0+18, y0+42), F, .95, col, 2, cv2.LINE_AA)
    cv2.putText(img, f'Total reps : {cum_reps[k]}', (x0+18, y0+90),
                F, .85, (255, 255, 255), 2, cv2.LINE_AA)
    if working:
        cv2.putText(img, f'Set reps   : {reps_in_set[k]}', (x0+18, y0+128),
                    F, .85, (80, 230, 80), 2, cv2.LINE_AA)
    with np.errstate(invalid='ignore'):
        ang = np.nanmean([S['angL_s'][k], S['angR_s'][k]])
    cv2.putText(img, f'Elbow  : {ang:.0f} do' if np.isfinite(ang) else 'Elbow  : --',
                (x0+18, y0+166), F, .7, (200, 200, 200), 2, cv2.LINE_AA)
    l, r = SIG_L[k], SIG_R[k]
    fmt = '{:+.2f}' if abs(P['sig_range'][1]) <= 2 else '{:.0f}'
    cv2.putText(img, ('L ' + fmt + '   R ' + fmt).format(l, r),
                (x0+18, y0+196), F, .7, (0, 220, 255), 2, cv2.LINE_AA)
    return x0, y0 + ph + 12, pw

In [ ]:
def frames_to_keep():
    """Mặt nạ frame nào được ghi ra. Bỏ đoạn nghỉ là cách giảm dung lượng rẻ nhất."""
    if not (RENDER_ONLY_SETS and SETS):
        return np.ones(TOTAL, bool)
    keep = np.zeros(TOTAL, bool)
    for st in SETS:
        a = int(max(0.0, st['start'] - RENDER_MARGIN_S) * FPS)
        b = int(min(TOTAL / FPS, st['end'] + RENDER_MARGIN_S) * FPS)
        keep[a:b] = True
    return keep


def render(src, dst, show_progress_every=2000):
    keep = frames_to_keep()
    # kích thước chẵn: libx264 yêu cầu chiều rộng/cao chia hết cho 2
    ow, oh = (int(W * RENDER_SCALE) // 2) * 2, (int(H * RENDER_SCALE) // 2) * 2
    print(f'Xuất {keep.sum()}/{TOTAL} frame ({keep.sum()/FPS:.0f}s trên {TOTAL/FPS:.0f}s) '
          f'ở {ow}x{oh}')

    cap = cv2.VideoCapture(src)
    out = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (ow, oh))
    assert out.isOpened(), 'Không mở được VideoWriter'

    # ánh xạ chỉ số frame -> chỉ số mẫu gần nhất
    lut = np.clip(np.searchsorted(idx, np.arange(TOTAL)), 0, len(idx) - 1)

    i, t0, flash, prev_k, n_out = 0, time.time(), 0, None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i >= len(keep) or not keep[i]:
            i += 1
            continue

        k  = lut[i]
        kp = kpts[k]

        if np.isfinite(kp[:, 0]).any():
            if BLUR_FACE:
                blur_face(frame, kp, W, H)
            draw_skeleton(frame, kp)

        px, py, pw = draw_hud(frame, k, W)
        if DRAW_PLOT:
            draw_trace(frame, k, (px, py, pw, 150))

        # nháy viền khi vừa đếm được 1 rep. So với frame ĐÃ GHI trước đó, không phải
        # i-1 — khi bỏ bớt frame thì i-1 có thể đã bị loại.
        if prev_k is not None and cum_reps[k] > cum_reps[prev_k]:
            flash = int(FPS * 0.25)
        prev_k = k
        if flash > 0:
            cv2.rectangle(frame, (0, 0), (W-1, H-1), (0, 255, 0), 14)
            flash -= 1

        # mốc thời gian là giờ TRONG VIDEO GỐC, để đối chiếu được với bảng set
        cv2.putText(frame, time.strftime('%M:%S', time.gmtime(i / FPS)),
                    (30, H - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2, cv2.LINE_AA)

        # thu nhỏ SAU khi vẽ: chữ vẽ ở cỡ đầy đủ rồi hạ mẫu vẫn đọc tốt,
        # nên không phải nhân tỉ lệ cho từng toạ độ vẽ
        out.write(cv2.resize(frame, (ow, oh), interpolation=cv2.INTER_AREA)
                  if (ow, oh) != (W, H) else frame)
        n_out += 1

        if i % show_progress_every == 0:
            el = time.time() - t0
            d  = max(i / TOTAL, 1e-9)
            print(f'  render {i}/{TOTAL} ({d*100:5.1f}%)  {el:.0f}s  '
                  f'còn ~{el/d-el:.0f}s', flush=True)
        i += 1

    cap.release(); out.release()
    print(f'Đã ghi {dst}: {n_out} frame, {os.path.getsize(dst)/1e6:.0f} MB, '
          f'{time.time()-t0:.0f}s')


if RENDER_VIDEO:
    render(VIDEO_PATH, OUT_VIDEO)
else:
    print('RENDER_VIDEO = False, bỏ qua.')

### Chuyển sang H.264 và xem ngay trong notebook

`mp4v` xuất được nhưng trình duyệt không phát, nên phải chuyển sang H.264.

Tab **Output** của Kaggle chỉ dựng được preview cho file nhỏ; gặp file vài trăm MB nó
báo *"Unable to show preview — Unexpected end of JSON input"*. Đó là lỗi của trình xem
trước, **không phải video hỏng** — tải về vẫn phát bình thường. Ô dưới in ra dung lượng
và nhúng luôn video vào notebook nếu đủ nhẹ, nên không cần tab Output nữa.

In [ ]:
from IPython.display import Video, display

H264_PATH = OUT_VIDEO.replace('.mp4', '_h264.mp4')
if RENDER_VIDEO and os.path.exists(OUT_VIDEO):
    rc = os.system(f'ffmpeg -y -loglevel error -i "{OUT_VIDEO}" -c:v libx264 '
                   f'-preset veryfast -crf {H264_CRF} -pix_fmt yuv420p "{H264_PATH}"')
    if rc == 0:
        os.remove(OUT_VIDEO)              # bỏ bản mp4v, chỉ giữ bản H.264 nhẹ hơn
        mb = os.path.getsize(H264_PATH) / 1e6
        print(f'{H264_PATH}  {mb:.1f} MB')
        if mb > 100:
            print('CẢNH BÁO: >100 MB, tab Output của Kaggle nhiều khả năng không '
                  'dựng nổi preview. Hạ RENDER_SCALE, bật RENDER_ONLY_SETS, '
                  'hoặc nâng H264_CRF rồi chạy lại ô render.')
        else:
            display(Video(H264_PATH, embed=(mb < 40), width=360))
    else:
        print('ffmpeg lỗi — dùng file mp4v:', OUT_VIDEO)

---
## Ghi chú khi chỉnh

| Triệu chứng | Chỉnh |
|---|---|
| Đếm thiếu rep (cuốn không hết tầm) | Hạ `REP_HI`, hạ `REP_MIN_AMP` |
| Đếm dư (rung tay lúc nghỉ cũng tính) | Nâng `REP_MIN_AMP`, nới `REP_HI`/`REP_LO` xa nhau hơn |
| Một nhịp bị đếm thành hai | Nâng `REP_MIN_PERIOD`, tăng `SMOOTH_WIN_S` |
| Một set bị cắt làm đôi | Nâng `SET_MAX_GAP_S` |
| Hai set dính làm một | Hạ `SET_MAX_GAP_S` |
| Tín hiệu đứt đoạn nhiều | Hạ `KP_CONF` xuống 0.35, hoặc đổi `MODEL_NAME` sang `yolo11m-pose.pt` |
| Hai tay cuốn cùng lúc bị tính thành 2 rep | Nâng `MERGE_WIN_S` |
| Cuốn xen kẽ bị gộp thành 1 rep | Hạ `MERGE_WIN_S` |
| Động tác lẻ bị tính thành set | Nâng `SET_MIN_REPS`, hạ `SET_MAX_CADENCE_S` |
| Chỉ đếm được 1 rep mỗi set | Ngưỡng đang đặt theo biên độ *tư thế* thay vì biên độ *rep* — chạy lại ô dò ngưỡng, nó chỉ lấy phân vị trong set |

Đổi bài tập khác thì chỉ cần thay tín hiệu trong `build_signals` — phần đếm rep và
tách set không phụ thuộc vào bài tập:

- **Squat**: góc gối `hông–gối–cổ chân`, hoặc chiều cao hông chuẩn hoá theo chiều dài đùi.
- **Push-up**: góc khuỷu `vai–khuỷu–cổ tay` (quay ngang thì vai rõ nên góc khuỷu ổn).
- **Nâng vai / vai trước**: độ nghiêng cánh tay trên `vai–khuỷu` thay cho cẳng tay.

Thêm bài mới = thêm một mục vào `PROFILES` và (nếu cần) một tín hiệu mới trong
`build_signals`. Phần đếm rep, ghép hai tay và tách set không phải sửa gì.

---
# Chạy hàng loạt cả thư mục

Đặt `BATCH_DIR` trỏ vào thư mục video rồi chạy ô dưới: notebook phân tích **mọi video
trong thư mục** bằng cùng một profile và xuất một bảng tổng hợp.

Toàn bộ bước phân tích tái sử dụng đúng các hàm đã dùng ở trên (`extract_keypoints`,
`build_signals`, `detect_reps`, `merge_arms`, `group_sets`) — không có nhánh code thứ hai
để lệch nhau. Ô này **không vẽ video** vì bảy video là quá nhiều; muốn xem video có chú
thích của một file cụ thể thì đặt `BATCH_DIR = None` và trỏ `VIDEO_PATH` vào file đó.

Cache keypoint đặt tên theo từng video nên chạy lại chỉ tốn thời gian cho video mới.

In [ ]:
def analyze_video(path, model=None, stride=STRIDE, verbose=True):
    """Phân tích một video -> dict kết quả. Dùng lại y nguyên các hàm ở trên."""
    tag = os.path.splitext(os.path.basename(path))[0][:40]
    npz = f'{OUT_DIR}/kpts_{EXERCISE}_{re.sub(r"[^A-Za-z0-9_-]", "_", tag)}.npz'

    if not cache_is_valid(npz, path, stride, MODEL_NAME):
        if model is None:
            model = YOLO(MODEL_NAME)
        i_, k_, b_, meta = extract_keypoints(path, model, stride=stride)
        np.savez_compressed(npz, idx=i_, kpts=k_, bconf=b_,
                            stride=stride, src=path, model=MODEL_NAME, **meta)

    d = np.load(npz)
    kp, ix = d['kpts'], d['idx']
    fps = float(d['fps'])
    tt = ix / fps
    sr = fps / float(d['stride'])
    if len(tt) < 20:
        return dict(video=os.path.basename(path), error='quá ít mẫu')

    Sv = build_signals(kp)
    gap = max(1, int(round(MAX_GAP_S * sr)))
    win = max(5, int(round(SMOOTH_WIN_S * sr)) | 1)
    key = {'forearm_tilt': 'fore', 'elbow_angle': 'ang',
           'wrist_above_nose': 'nosew'}[P['signal']]
    sL = smooth(interp_short_gaps(Sv[key + 'L'], gap), win)
    sR = smooth(interp_short_gaps(Sv[key + 'R'], gap), win)

    reps = merge_arms(detect_reps(tt, sL, 'L'), detect_reps(tt, sR, 'R'))
    sets = group_sets(reps)
    in_sets = sum(s['n_reps'] for s in sets)
    cad = [np.median(np.diff([r['t_top'] for r in s['reps']])) for s in sets
           if len(s['reps']) > 1]
    # tỉ lệ mẫu dùng được: thước đo chất lượng pose, quan trọng hơn ta tưởng
    valid = float(np.mean(np.isfinite(sL) | np.isfinite(sR)))

    if verbose:
        print(f'  -> {in_sets} rep / {len(sets)} set, tín hiệu dùng được '
              f'{valid*100:.0f}% mẫu', flush=True)

    return dict(video=os.path.basename(path), thoi_luong_s=round(tt[-1], 1),
                fps=round(fps, 2), so_mau=len(tt), tin_hieu_ok=round(valid, 3),
                rep=in_sets, rep_le=len(reps) - in_sets, set=len(sets),
                nhip_s=round(float(np.median(cad)), 2) if cad else np.nan,
                tap_s=round(sum(s['dur'] for s in sets), 1))


if BATCH_FILES:
    _model = YOLO(MODEL_NAME)
    rows, t_all = [], time.time()
    for n, f in enumerate(BATCH_FILES, 1):
        print(f'[{n}/{len(BATCH_FILES)}] {os.path.basename(f)}', flush=True)
        try:
            rows.append(analyze_video(f, _model))
        except Exception as e:
            print('   LỖI:', e)
            rows.append(dict(video=os.path.basename(f), error=str(e)[:80]))
    df_batch = pd.DataFrame(rows)
    df_batch.to_csv(f'{OUT_DIR}/batch_{EXERCISE}.csv', index=False)
    display(df_batch)
    if 'rep' in df_batch:
        print(f"\nTỔNG: {int(df_batch['rep'].fillna(0).sum())} rep trên "
              f"{len(BATCH_FILES)} video, {time.time()-t_all:.0f}s")
else:
    print('BATCH_DIR chưa đặt -> bỏ qua. Đặt BATCH_DIR trong ô Cấu hình để chạy hàng loạt.')

### Đọc cột `tin_hieu_ok` trước khi tin cột `rep`

`tin_hieu_ok` là tỉ lệ mẫu mà tín hiệu tính được. Nó là **cảnh báo sớm** cho biết số rep
có đáng tin không, và cần nhìn trước tiên.

Trên bộ video push-up đo được, chất lượng pose chênh nhau rất xa dù cùng một góc quay:

| Video | Đủ 6 keypoint thân trên |
|---|---|
| `IMG_5087.MOV` | **98.8 %** |
| `100pushup.mp4` | **72.2 %** |

Và điều tệ nhất là **chỗ hỏng không rải đều**: nó dồn đúng vào **đáy mỗi rep**, khi đầu
lấp đầy khung và người bị khung cắt. Đo trên `100pushup.mp4`: trung bình **1.8 lần mất
keypoint mỗi rep**, mỗi lần ~0.21 s. Đáy chính là thứ định nghĩa một rep, nên mất ở đó
là mất rep.

Dùng model to hơn **không cứu được**: `yolo11x-pose` ở `imgsz=960` cũng chỉ đủ keypoint ở
1/8 frame đáy. Đây là giới hạn của góc quay, không phải của dung lượng model — camera
đặt rất thấp ngay trước mặt là tư thế hiếm trong dữ liệu huấn luyện COCO.

Cách sửa nằm ở **khâu quay, không phải khâu code**: đặt máy **ngang hông, quay từ bên
sườn**, lùi đủ xa để cả người nằm trọn trong khung. Cùng thuật toán này khi đó chạy ổn
hơn hẳn.

---
## Vì sao push-up dùng góc khuỷu

Ngược với dự đoán ban đầu: tôi tưởng góc nhìn trực diện sẽ làm góc khuỷu vô dụng vì
**rút ngắn phối cảnh**. Đo thực tế thì ngược lại — camera đặt thấp ngay phía trước nên
cánh tay vung gần như song song mặt phẳng ảnh, và góc khuỷu dao động rất rõ 70° ↔ 170°.

Bốn ứng viên khác đều đếm đúng trên một video, nên tiêu chí chọn không phải độ chính xác
mà là **độ rộng của vùng ngưỡng ổn định** — quét ngưỡng cố định dùng chung cho hai video
có chất lượng pose rất khác nhau:

| Tín hiệu | Số tổ hợp ngưỡng đúng trên **cả hai** video |
|---|---|
| **Góc khuỷu** | **24** |
| `(y_cổtay − y_vai) / bề rộng vai` | 6 |

Vùng ổn định rộng gấp 4 lần nghĩa là ngưỡng không nằm trên lưỡi dao. Góc khuỷu còn có
một lợi thế mà tín hiệu tỉ lệ không có: **đơn vị vật lý là độ**, nên ngưỡng chuyển được
sang video khác, người khác, khoảng cách camera khác mà không phải dò lại.

Một cái bẫy đã gặp và đã xử lý: chuẩn hoá theo bề rộng vai bị **nổ tung khi pose sập** —
bề rộng vai tụt về gần 0 làm phép chia bắn tín hiệu lên 30×. `build_signals` giờ loại
mọi mẫu có bề rộng vai nằm ngoài 0.45–2.2 lần trung vị (chiếm 13.3 % ở video khó nhất).

## Vì sao hít đất đếm trên TRUNG BÌNH hai tay

Hai bài tạ đếm từng tay rồi gộp, vì chúng có thể tập xen kẽ hoặc một tay. Hít đất thì
**không thể**: hai tay bị ràng buộc cứng cùng pha. Đếm tách rồi gộp lại thành ra khuếch
đại một lỗi cụ thể của pose — **gán nhầm tay trái/phải**. Khi đó tay "trái" và tay "phải"
lệch nhau 0.5–0.7 s, vượt `MERGE_WIN_S` nên không được gộp, và mỗi lần nhầm là sinh thêm
một rep ma.

Đo trên `100pushup.mp4`, cùng bộ ngưỡng:

| Cách đếm | Cửa sổ khó (mắt đếm 3) | Cửa sổ sạch (mắt đếm 5) | Toàn video |
|---|---|---|---|
| Tách tay rồi gộp | **5** ✗ | 5 ✓ | 120 |
| Trung bình hai tay | 4 ✗ | 5 ✓ | 100 |
| Trung bình + `min_period=0.7` | **3** ✓ | **5** ✓ | **97** |

Bài học: `MERGE_WIN_S` không cứu được kiểu lỗi này, vì độ lệch do gán nhầm tay
(0.5–0.7 s) rơi đúng vào vùng xám giữa "cùng nhịp" và "hai nhịp khác nhau". Phải đổi
mô hình đếm chứ không phải nới ngưỡng.